In [3]:
import pandas as pd

def read_player_stats(path):
    """Đọc CSV theo từng dòng: thử UTF-8 trước, dòng nào lỗi (bị lưu bằng Latin-1)
    thì fallback sang Latin-1 cho đúng dòng đó. Tránh lỗi font kiểu 'JoÃ£o' do
    đọc toàn bộ file bằng encoding='latin-1' trước đây."""
    from io import StringIO
    lines = []
    with open(path, 'rb') as f:
        for line in f:
            try:
                lines.append(line.decode('utf-8'))
            except UnicodeDecodeError:
                lines.append(line.decode('latin-1'))
    return pd.read_csv(StringIO(''.join(lines)))

df = read_player_stats('../data/player_stats.csv')

print(f"Kích thước bộ dữ liệu ban đầu: {df.shape}")

missing_data = df.isnull().sum()
print("\nDanh sách các cột bị khuyết dữ liệu:")
print(missing_data[missing_data > 0])

df_cleaned = df.drop(columns=['marking'])
print(f"\nKích thước bộ dữ liệu sau khi làm sạch: {df_cleaned.shape}")

superstars = df_cleaned[df_cleaned['player'].str.contains('Ronaldo|Mbappé', na=False, case=False)]

print("\n--- Bảng chỉ số của các siêu sao ---")
# Chọn ra một vài thuộc tính cốt lõi để hiển thị: tên, câu lạc bộ, tốc độ, dứt điểm, sút xa, giá trị
cols_to_show = ['player', 'club', 'sprint_speed', 'finishing', 'long_shots', 'value']
display(superstars[cols_to_show])

Kích thước bộ dữ liệu ban đầu: (5682, 41)

Danh sách các cột bị khuyết dữ liệu:
marking    5682
dtype: int64

Kích thước bộ dữ liệu sau khi làm sạch: (5682, 40)

--- Bảng chỉ số của các siêu sao ---


,player,club,sprint_speed,finishing,long_shots,value
2381,Ronaldo Damus,GIF Sundsvall,77,63,54,$400.00
3401,Ronaldo Cabrais,Brazil,87,74,78,$35.000.000
5229,Kylian Mbappé,Paris SG,97,93,82,$153.500.000
5365,Ronaldo Vieira,Torino,76,53,62,$1.900.000
5675,Kylian Mbappé,Paris SG,97,93,82,$153.500.000
5680,Cristiano Ronaldo,Al Nassr,82,91,88,$31.000.000


In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Đọc dữ liệu và loại bỏ cột 'marking'
def read_player_stats(path):
    """Đọc CSV theo từng dòng: thử UTF-8 trước, dòng nào lỗi (bị lưu bằng Latin-1)
    thì fallback sang Latin-1 cho đúng dòng đó. Tránh lỗi font kiểu 'JoÃ£o' do
    đọc toàn bộ file bằng encoding='latin-1' trước đây."""
    from io import StringIO
    lines = []
    with open(path, 'rb') as f:
        for line in f:
            try:
                lines.append(line.decode('utf-8'))
            except UnicodeDecodeError:
                lines.append(line.decode('latin-1'))
    return pd.read_csv(StringIO(''.join(lines)))

df = read_player_stats('../data/player_stats.csv')
df_cleaned = df.drop(columns=['marking'])

# 2. Xử lý cột 'value'
# Loại bỏ ký hiệu '$' và dấu chấm (phân cách hàng nghìn), chuyển đổi thành kiểu float
df_cleaned['value_numeric'] = df_cleaned['value'].replace('[\$\.]', '', regex=True).astype(float)

# 3. Chuẩn hóa dữ liệu bằng MinMaxScaler
features_to_normalize = ['sprint_speed', 'finishing', 'long_shots', 'value_numeric']

scaler = MinMaxScaler()
df_scaled = df_cleaned.copy()
df_scaled[features_to_normalize] = scaler.fit_transform(df_cleaned[features_to_normalize])

# 4. Hiển thị kết quả chuẩn hóa
superstars_scaled = df_scaled[df_scaled['player'].str.contains('Ronaldo|Mbappé', na=False, case=False)]
print("\n--- Bảng chỉ số của các siêu sao sau khi chuẩn hóa ---")
cols_to_show = ['player', 'club'] + features_to_normalize
display(superstars_scaled[cols_to_show])

<>:24: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:24: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
/tmp/ipykernel_7050/4207146017.py:24: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  df_cleaned['value_numeric'] = df_cleaned['value'].replace('[\$\.]', '', regex=True).astype(float)



--- Bảng chỉ số của các siêu sao sau khi chuẩn hóa ---


,player,club,sprint_speed,finishing,long_shots,value_numeric
2381,Ronaldo Damus,GIF Sundsvall,0.761905,0.655556,0.569767,0.000258
3401,Ronaldo Cabrais,Brazil,0.880952,0.777778,0.848837,0.228011
5229,Kylian Mbappé,Paris SG,1.000000,0.988889,0.895349,1.000000
5365,Ronaldo Vieira,Torino,0.750000,0.544444,0.662791,0.012375
5675,Kylian Mbappé,Paris SG,1.000000,0.988889,0.895349,1.000000
5680,Cristiano Ronaldo,Al Nassr,0.821429,0.966667,0.965116,0.201952


Data Integration: Xử lý thành công lỗi UnicodeDecodeError bằng cách ép kiểu mã hóa latin-1, giúp hệ thống đọc mượt mà các tên cầu thủ có ký tự đặc biệt.
Data Cleaning (Làm sạch): Dùng code để tự động quét và phát hiện cột marking bị khuyết 100% dữ liệu, sau đó mạnh dạn loại bỏ (drop) để tránh nhiễu (noise) cho thuật toán.  
Data Transformation (Biến đổi): Áp dụng kỹ thuật chuẩn hóa Min-Max (MinMaxScaler) để đưa các chỉ số có khoảng giá trị chênh lệch lớn về cùng hệ quy chiếu [0, 1], đảm bảo tính công bằng khi tính khoảng cách thuật toán.  